In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/nba_features.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
print(df[['team', 'date', 'opponent', 'result']].head(10))

Shape: (35995, 37)
Date range: 2012-10-05 00:00:00 to 2025-06-22 00:00:00
  team       date opponent  result
0  FBU 2012-10-05      BOS       1
1  BOS 2012-10-05      FBU       0
2  MPS 2012-10-06      SAS       0
3  ALB 2012-10-06      DAL       0
4  MEM 2012-10-06      RMD       1
5  LAC 2012-10-06      DEN       0
6  SAS 2012-10-06      MPS       1
7  RMD 2012-10-06      MEM       0
8  DAL 2012-10-06      ALB       1
9  DEN 2012-10-06      LAC       1


In [9]:
NBA_TEAMS = [
    'ATL', 'BOS', 'BKN', 'CHI', 'CHA', 'CLE', 'DAL', 'DEN', 'DET',
    'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN',
    'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS',
    'TOR', 'UTA', 'WAS'
]

df = df[df['team'].isin(NBA_TEAMS)].copy()
df = df[df['SEASON_ID'].astype(str).str.startswith('2')].copy()
df = df.reset_index(drop=True)

print(f"After filtering: {len(df)} games, {df['team'].nunique()} teams")

After filtering: 31254 games, 30 teams


In [10]:
def calculate_elo(df, k=20, initial=1500):
    """
    Calculate ELO rating for every team before each game.
    k  = how fast ratings change (20 is standard)
    initial = starting ELO for all teams (1500 is standard)
    """
    
    elo = {}
    for team in df['team'].unique():
        elo[team] = initial

    team_elo_before = []  
    opp_elo_before  = []

    for _, row in df.iterrows():
        team = row['team']
        opp  = row['opponent']

        
        t_elo = elo.get(team, initial)
        o_elo = elo.get(opp,  initial)

        
        team_elo_before.append(t_elo)
        opp_elo_before.append(o_elo)

        
        expected = 1 / (1 + 10 ** ((o_elo - t_elo) / 400))
        actual   = row['result']  

        
        elo[team] = t_elo + k * (actual - expected)
        elo[opp]  = o_elo + k * ((1 - actual) - (1 - expected))

    df['team_elo']    = team_elo_before
    df['opp_elo']     = opp_elo_before
    df['elo_diff']    = df['team_elo'] - df['opp_elo']

    return df

df = calculate_elo(df)
print("ELO added!")
print(df[['team', 'date', 'team_elo', 'opp_elo', 'elo_diff', 'result']].head(10))

ELO added!
  team       date     team_elo      opp_elo   elo_diff  result
0  DAL 2012-10-30  1500.000000  1500.000000   0.000000       1
1  MIA 2012-10-30  1500.000000  1500.000000   0.000000       1
2  CLE 2012-10-30  1500.000000  1500.000000   0.000000       1
3  WAS 2012-10-30  1490.000000  1510.000000 -20.000000       0
4  LAL 2012-10-30  1490.000000  1510.000000 -20.000000       0
5  BOS 2012-10-30  1490.000000  1510.000000 -20.000000       0
6  LAL 2012-10-31  1480.575011  1500.000000 -19.424989       0
7  SAC 2012-10-31  1500.000000  1500.000000   0.000000       0
8  POR 2012-10-31  1509.441486  1471.133526  38.307960       1
9  PHX 2012-10-31  1500.000000  1500.000000   0.000000       0


In [11]:
def add_h2h(df):
    """Win rate of team vs this specific opponent historically."""
    h2h_rates = []

    for i, row in df.iterrows():
        team = row['team']
        opp  = row['opponent']
        date = row['date']

        
        past = df[
            (df['team'] == team) &
            (df['opponent'] == opp) &
            (df['date'] < date)
        ].tail(10)  

        rate = past['result'].mean() if len(past) > 0 else 0.5
        h2h_rates.append(rate)

        
        if i % 5000 == 0:
            print(f"  Processing row {i}/{len(df)}...")

    df['h2h_win_rate'] = h2h_rates
    return df

print("Calculating H2H rates (this takes 3-5 minutes)...")
df = add_h2h(df)
print("Done!")
print(df[['team', 'opponent', 'h2h_win_rate']].head(10))

Calculating H2H rates (this takes 3-5 minutes)...
  Processing row 0/31254...
  Processing row 5000/31254...
  Processing row 10000/31254...
  Processing row 15000/31254...
  Processing row 20000/31254...
  Processing row 25000/31254...
  Processing row 30000/31254...
Done!
  team opponent  h2h_win_rate
0  DAL      LAL           0.5
1  MIA      BOS           0.5
2  CLE      WAS           0.5
3  WAS      CLE           0.5
4  LAL      DAL           0.5
5  BOS      MIA           0.5
6  LAL      POR           0.5
7  SAC      CHI           0.5
8  POR      LAL           0.5
9  PHX      GSW           0.5


In [12]:
df.to_csv('../data/processed/nba_final.csv', index=False)
print(f"Saved! {len(df)} rows, {len(df.columns)} columns")
print("\nFinal feature list:")
print(df.columns.tolist())

Saved! 31254 rows, 41 columns

Final feature list:
['SEASON_ID', 'TEAM_ID', 'team', 'TEAM_NAME', 'GAME_ID', 'date', 'matchup', 'result_str', 'MIN', 'team_score', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'result', 'is_home', 'opponent', 'roll_pts_scored', 'roll_plus_minus', 'roll_win_rate', 'roll_fg_pct', 'days_rest', 'streak', 'team_elo', 'opp_elo', 'elo_diff', 'h2h_win_rate']
